# 43 · Campos vazios: quem entra no k-means e com quais variáveis

O k-means não aceita linha com valor ausente. Na tabela master os vazios têm significado (não
respondeu o onboarding, não informou a idade, não treinou), então nenhum é preenchido:

1. **Variáveis que saem do k-means**: `idade_no_corte`, `dias_por_semana_declarado` e `aderencia_meta_cap`.
   Continuam na tabela e servem para descrever os grupos depois de formados.
2. **Segmento por regra**: assinante ativo sem treino nos 90 dias até o corte é rotulado "inativo em
   90 dias" e não entra no k-means. O k-means roda nos assinantes ativos que treinaram.
3. **Fora do escopo**: quem não é assinante ativo no corte.

Entra uma coluna nova, `segmento_regra`. Nada é apagado nem preenchido.
Ordem de execução: 39 → 41 → 42 → 43.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master lida:", m.shape)
if "segmento_regra" in m.columns: m = m.drop(columns=["segmento_regra"])
orig = m.copy()
Z = [c for c in m.columns if c.endswith("_z")]
IND = ["pedala_90d", "treinou_acompanhado_90d", "abriu_chamado", "tem_relogio"]
assert len(Z) == 23 and all(c in m.columns for c in IND), "rode os notebooks 41 e 42 antes"
FORA = {"idade_no_corte_z": "897 ativos sem nascimento informado; idade não separou ativos, engajamento nem tipo de treino nas análises",
        "dias_por_semana_declarado_z": "1.491 ativos não responderam a meta no onboarding; fica para descrever os grupos",
        "aderencia_meta_cap_z": "depende da meta declarada (1.491 ativos sem resposta) e repete a frequência de treino"}
ENTRA = [c for c in Z if c not in FORA] + IND
print("padronizadas:", len(Z), "| saem:", len(FORA), "| entram:", len(ENTRA), "(", len(ENTRA) - len(IND), "padronizadas +", len(IND), "indicadores )")

tabela master lida: (63929, 118)
padronizadas: 23 | saem: 3 | entram: 24 ( 20 padronizadas + 4 indicadores )


## 1. Vazios entre os assinantes ativos, antes

In [2]:
at = m[m.ativa_no_corte]
v = at[Z + IND].isna().sum()
print("assinantes ativos:", len(at))
print(v[v > 0].sort_values(ascending=False).to_frame("ativos sem valor").assign(pct=lambda d: d["ativos sem valor"] / len(at) * 100))
print("ativos com as 23 padronizadas e os 4 indicadores preenchidos:", int(at[Z + IND].notna().all(axis=1).sum()), f"({at[Z + IND].notna().all(axis=1).mean():.1%})")
sem = (at.treinos_90d == 0).fillna(False).astype(bool)
print(); print("ativos sem treino nos 90 dias:", int(sem.sum()))
print("  plano:", at[sem].plano.value_counts().to_dict(), "| nível:", at[sem].nivel_engajamento.value_counts().to_dict())
print("  sem peso_tipo_principal:", int(at[sem].peso_tipo_principal.isna().sum()), "| sem dias_sem_treinar:", int(at[sem].dias_sem_treinar.isna().sum()))
print("ativos que treinaram e não têm dias_sem_treinar ou peso_tipo_principal:", int((at[~sem].dias_sem_treinar.isna() | at[~sem].peso_tipo_principal.isna()).sum()))

assinantes ativos: 14753
                             ativos sem valor   pct
aderencia_meta_cap_z                     1491 10.11
dias_por_semana_declarado_z              1491 10.11
idade_no_corte_z                          897  6.08
peso_tipo_principal_z                     754  5.11
dias_sem_treinar_log_z                    176  1.19
ativos com as 23 padronizadas e os 4 indicadores preenchidos: 11811 (80.1%)

ativos sem treino nos 90 dias: 754
  plano: {'anual': 751, 'mensal': 3} | nível: {'possível churn': 749, 'médio': 5}
  sem peso_tipo_principal: 754 | sem dias_sem_treinar: 176
ativos que treinaram e não têm dias_sem_treinar ou peso_tipo_principal: 0


## 2. Segmento por regra

In [3]:
m["segmento_regra"] = np.select([(~m.ativa_no_corte).to_numpy(bool), (m.treinos_90d == 0).fillna(False).to_numpy(bool)], ["fora do escopo (não é assinante ativo)", "inativo em 90 dias"], "elegível ao k-means")
t = m.segmento_regra.value_counts().to_frame("contas"); t["%"] = t.contas / t.contas.sum() * 100; print(t)
print(); print(pd.crosstab(m.situacao, m.segmento_regra, margins=True))
el = m[m.segmento_regra == "elegível ao k-means"]
ina = m[m.segmento_regra == "inativo em 90 dias"]
print(); print("perfil dos dois grupos de assinantes ativos")
cmp = pd.DataFrame({"elegível ao k-means": [len(el), (el.plano == "anual").mean() * 100, el.treinos_90d.astype(float).mean(), (el.nivel_engajamento == "alto").mean() * 100, el.renovacao_desligada.mean() * 100, el.meses_de_casa.median()],
                    "inativo em 90 dias": [len(ina), (ina.plano == "anual").mean() * 100, ina.treinos_90d.astype(float).mean(), (ina.nivel_engajamento == "possível churn").mean() * 100, ina.renovacao_desligada.mean() * 100, ina.meses_de_casa.median()]},
                   index=["contas", "% anual", "treinos em 90 dias (média)", "% no nível alto | % em possível churn", "% renovação desligada", "meses de casa (mediana)"])
print(cmp)

                                        contas     %
segmento_regra                                      
fora do escopo (não é assinante ativo)   49176 76.92
elegível ao k-means                      13999 21.90
inativo em 90 dias                         754  1.18

segmento_regra              elegível ao k-means  fora do escopo (não é assinante ativo)  inativo em 90 dias    All
situacao                                                                                                          
assinante ativo                           13999                                       0                 754  14753
em teste grátis                               0                                     470                   0    470
ex-assinante                                  0                                   17830                   0  17830
nunca iniciou teste                           0                                   11733                   0  11733
teste expirou, não assinou                  

## 3. Conferências

In [4]:
print("linhas:", len(m), "(antes:", len(orig), ") | colunas:", orig.shape[1], "->", m.shape[1])
iguais = all(m[c].equals(orig[c]) for c in orig.columns); print("colunas existentes intactas:", iguais)
assert iguais and len(m) == len(orig) and m.user_id.is_unique
vaz = el[ENTRA].isna().sum()
print("elegíveis ao k-means:", len(el), "| vazios nas", len(ENTRA), "variáveis que entram:", int(vaz.sum()))
assert vaz.sum() == 0
print("vazios dos elegíveis nas 3 variáveis que saem:", el[list(FORA)].isna().sum().to_dict())
print("linhas completas se as 3 variáveis ficassem:", int(el[Z + IND].notna().all(axis=1).sum()), "-> com a decisão:", len(el), "| ganho:", len(el) - int(el[Z + IND].notna().all(axis=1).sum()))
print(); print("escala das variáveis que entram, entre os elegíveis (a padronização foi feita nos 14.753 ativos)")
d = el[[c for c in ENTRA if c.endswith('_z')]].describe().T[["mean", "std", "min", "max"]]; print(d)
print(); print("indicadores entre os elegíveis (% sim):", (el[IND].mean() * 100).round(1).to_dict())

linhas: 63929 (antes: 63929 ) | colunas: 118 -> 119


colunas existentes intactas: True


elegíveis ao k-means: 13999 | vazios nas 24 variáveis que entram: 0
vazios dos elegíveis nas 3 variáveis que saem: {'idade_no_corte_z': 851, 'dias_por_semana_declarado_z': 1414, 'aderencia_meta_cap_z': 1414}
linhas completas se as 3 variáveis ficassem: 11811 -> com a decisão: 13999 | ganho: 2188

escala das variáveis que entram, entre os elegíveis (a padronização foi feita nos 14.753 ativos)
                             mean  std   min  max
duracao_dias_log_z          -0.03 1.01 -4.22 1.77
n_cobrancas_aprovadas_log_z  0.03 1.02 -0.85 2.43
receita_liquida_z           -0.01 1.01 -1.52 2.99
treinos_7d_z                 0.05 1.00 -1.00 6.60
treinos_30d_z                0.07 0.98 -1.25 6.93
treinos_90d_cap_z            0.07 0.97 -1.30 3.17
dias_ativos_90d_z            0.08 0.96 -1.42 3.89
minutos_90d_cap_z            0.07 0.98 -1.28 3.43
dias_sem_treinar_log_z      -0.11 0.86 -1.08 2.29
n_tipos_z                    0.14 0.82 -1.85 1.07
peso_tipo_principal_z        0.00 1.00 -2.15 1.91
trein

## 4. Lista de variáveis, dicionário e gravação

In [5]:
lista = pd.DataFrame([{"variavel": c, "entra_no_kmeans": c in ENTRA, "tipo": "indicador (0/1)" if c in IND else "padronizada (z-score)",
                       "motivo_se_fora": FORA.get(c, ""), "vazios_elegiveis": int(el[c].isna().sum()), "vazios_ativos": int(at[c].isna().sum())} for c in Z + IND])
lista.to_csv(S / "tabela_master_variaveis_kmeans.csv", index=False, encoding="utf-8-sig")
print(lista.to_string())

dic = pd.read_csv(S / "tabela_master_dicionario.csv")
dic = dic[dic.coluna != "segmento_regra"][["coluna", "grupo", "descricao", "fonte"]]
dic = pd.concat([dic, pd.DataFrame([("segmento_regra", "segmentação", "elegível ao k-means (assinante ativo que treinou nos 90 dias até o corte), inativo em 90 dias (assinante ativo sem treino na janela; segmento definido por regra) ou fora do escopo (não é assinante ativo)", "derivada")], columns=["coluna", "grupo", "descricao", "fonte"])], ignore_index=True)
assert list(dic.coluna) == list(m.columns), "dicionário e tabela fora de ordem"
dic["tipo"] = [str(m[c].dtype) for c in dic.coluna]
dic["preenchidas"] = [int(m[c].notna().sum()) for c in dic.coluna]
dic["% preenchido"] = (dic.preenchidas / len(m) * 100).round(1)
dic["% preenchido (ativos)"] = [round(m.loc[m.ativa_no_corte, c].notna().mean() * 100, 1) for c in dic.coluna]
m.to_parquet(P / "tabela_master.parquet", index=False)
m.to_csv(S / "tabela_master.csv", index=False, encoding="utf-8-sig")
dic.to_csv(S / "tabela_master_dicionario.csv", index=False, encoding="utf-8-sig")
print(); print("gravado:", m.shape)

                       variavel  entra_no_kmeans                   tipo                                                                                              motivo_se_fora  vazios_elegiveis  vazios_ativos
0              idade_no_corte_z            False  padronizada (z-score)  897 ativos sem nascimento informado; idade não separou ativos, engajamento nem tipo de treino nas análises               851            897
1            duracao_dias_log_z             True  padronizada (z-score)                                                                                                                             0              0
2   n_cobrancas_aprovadas_log_z             True  padronizada (z-score)                                                                                                                             0              0
3             receita_liquida_z             True  padronizada (z-score)                                                                             


gravado: (63929, 119)
